# ML Pipelines: Deploying ML Models into Production
Author: Tomasz Kanas

Creating best possible ML model is important, but in real-life applications, it is not enough. Our models, as any other software, need to be maintained in the organization for long time. They will most likely evolve over time, as will the data they are being trained on. For this reason, the models ment for production should be developed using best software engineering practices and the whole training, deploying and serving pipeline should be reliable, well tested and give reproducible results.

The typical ML pipeline consists of at least those stages:

- Data generation
- Data ingestion
- Data transformation
- Serving

On the other hand, typical training pipeline looks similar:

- Data generation
- Data ingestion
- Data validation
- Data transformation
- Model training and tuning
- Model push into production

As you see, there is a lot of space for redundancy and errors here, especially when such pipelines are developed and maintained by several people, and contain manual steps. For this reason it is important to automate and test every step of the pipeline as well as whole pipeline end-to-end.

It is also a good idea to use an existing tool to streamline the developement of such pipeline. In this scenario we will use [Kedro](https://kedro.org/), an open-source framework for developing and releasing ML pipelines. There are several such libraries, alternatives include:
  - [ZenML](https://www.zenml.io/)
  - [Metaflow](https://metaflow.org/)
  - [TorchX](https://pytorch.org/torchx/latest/)



## Kedro installation and example

Kedro has fixed file structure and is intended to use with an IDE, not notebook, so during this scenario you will be working locally on your computer.

Start from creating a virtualenv and installing Kedro:
```
pip install kedro
```

Kedro offers several starter projects, so that you don't start from the blank slate. In this scenario we will use the `astro-airflow-iris` starter:
```
kedro new --starter=astro-airflow-iris
```
Enter a project name (e.g. "BML"), enter the newly created directory and install the requirements:
```
pip install -r requirements.txt
```
Now run the project:
```
kedro run
```
You should see that Kedro did run 4 tasks and finished training with around 90% accuracy.

Let's continue exploring the project. Kedro comes with a visualization tool `kedro-viz`. Install it and run:
```
pip install kedro-viz
kedro viz
```
It should open a browser tab with a visualization of the data pipeline (URL: http://127.0.0.1:4141). Take a look around the visualization, you can click most things there to get more information.

There are many methods of releasing model into production, including packaging it as a Python package and running on a baremetal or VM, or packaging it into a Docker or Kubernetes image. We will show how to release Kedro project into the Apache Airflow.

Apache Airflow is an orchestrator used by many companies to manage and run various processing pipelines. We firstly have to install and run Airflow locally:
```
pip install apache-airflow kedro-airflow
airflow standalone
```
The dashboard should be available under http://localhost:8080. To generate an Airflow DAG from the Kedro project run command:
```
kedro airflow create --target-dir=dags/
```
It will create a `dags` directory and a Python script inside. You need to copy this file into and `~/airflow/dags` directory (which is a default directory for storing Airflow DAGs and should be created when you started Airflow).

After some time (you can also restart Airflow if it takes too long to update), you should be able to see (and run) your DAG in the Airflow UI.

## Kedro project files

Let's now explore the contents of the project. In the main directory you should see numerous files - the purpose of most of them should be self-explaining: Dockerfile is a template to release your project to Docker, pyproject.ml is for releasing it as a Python package etc. For now those files aren't important for us. There is a `README.md` file which is a good guide how to use the template, you might have want to take a look at it.

### Configuration

Let's now explore the `conf` directory. It again contains a `README.md` with explanaisions, along with the `logging.yml` file - a default logging configuration (which is fine for us). Apart from that it contains a `base` and `local` directories, those are environments: `base` is for general configuration, while `local` is for the config for the local developement process (e.g. security keys, IDE configuration). If you need to have more environments (like `test`, `qa` or `prod`), you just need to create more directories here.

In the `base` directory there are two files: `catalog.yml` and `parameters.yml`. The `catalog.yml` is the file in which you define your datasets. The default file contains some instructions and examples how to do that. Take a look at it.

The `parameters.yml` is the file where you store the parameters of your pipeline, like number of epochs, learning rate, batch size etc.

Currently we have only one defined dataset: `example_iris_data`, but in the visualization you should see several intermediate datasets. Let's say, that we would like to save the weights of our model after training. In the visualization, you can see that the weights are defined as `example_model`, so you can just add:
```
example_model:
  type: pickle.PickleDataset
  filepath: data/06_models/model.pickle
```
to the `catalog.yml` file. Now run `kedro run`. After that you should be able to see the `data/06_models/model.pickle` file. When it comes to the model weights, it is a good practice, to keep also previous versions of your model, in case the new versions are faulty (then you can quickly fall-back to the old version in the production when you look for a proper fix). To do this, you can just add
```
  versioned: true
```
to the `example_model` definition. If you would now run `kedro run` it would fail - it can't just overwrite your dataset, you need to delete the `data/06_models/model.pickle` manually, and then run `kedro run`. After it finishes, you should see that `data/06_models/model.pickle` is a directory containing a subdirectory which name is a timestamp of the run. The final `model.pickle` file is inside this subdirectory.

### Sources

Let's now investigate the sources of the project in the `src` directory. You should see the `pipeline_registry.py` and `settings.py` files, as well as a `pipelines` directory containing two directories: `data_engineering` and `data_science`. This project indeed contains 2 pipelines. The `pipeline_registry.py` is a file that defines a default pipeline, that contains all defined pipelines - that's why when we run Kedro, whole process executes. If you want to run just a single pipeline you can run e.g. a command `kedro run --pipeline=data_science`.

Each pipeline contains a `README.md` file describing the pipeline, a `nodes.py` file containing the actual code of the pipeline, where each step is just a Python function, and a `pipeline.py` file with the pipeline definition.

Let's now take a look at the `data_science/pipeline.py` file:

In [ ]:
from kedro.pipeline import node, pipeline
from .nodes import predict, report_accuracy, train_model

def create_pipeline(**kwargs):
    return pipeline(
        [
            node(
                train_model,
                ["example_train_x", "example_train_y", "parameters"],
                "example_model",
                name="train",
            ),
            node(
                predict,
                dict(model="example_model", test_x="example_test_x"),
                "example_predictions",
                name="predict",
            ),
            node(
                report_accuracy,
                ["example_predictions", "example_test_y"],
                None,
                name="report",
            ),
        ]
    )


The pipeline is created from the list of nodes, each node executes a Python function (imported from the `nodes.py` file). Then you need to specify the names of the input datasets (either as a list or a dictionary) and the name(s) of the output datasets (None if no output). Finally you specify a name, that is visible in the visualization.

And that's it! This is enough work to create a simple ML pipeline. Take a look at the remaining files in the `src` directory. Note, that this is a simple example provided by Kedro, so it doesn't use any libraries apart from `numpy` and `pandas`, but in general you can use any Python code in your pipeline, including `scikit-learn` and `PyTorch`.

### A little refactor
The code provided in the example is very minimal. Let us now refactor it a bit. A good practice is to keep the features you are using in the configuration, so that it is easy to change them at any point. Moreover we would also like to have repeatability in our experiments, so we will save a seed to the random number generator in the configuration.

Change the `conf/base/parameters.yml` file to following:
```
model_options:
    test_data_ratio: 0.2
    num_train_iter: 10000
    learning_rate: 0.01
    random_state: 42
    features:
        - sepal_length
        - sepal_width
        - petal_length
        - petal_width
```

Now let's refactor the `data_engineering/nodes.py` file in a following way:

In [ ]:
from typing import Any

import logging
import pandas as pd
from sklearn.model_selection import train_test_split

def split_data(data: pd.DataFrame, parameters: dict[str, Any]) -> dict[str, Any]:
    log = logging.getLogger(__name__)
    log.info(data.columns)
    classes = sorted(data["species"].unique())
    # One-hot encoding for the target variable
    data = pd.get_dummies(data, columns=["species"], prefix="", prefix_sep="")

    # Split to training and testing data
    X = data[parameters["features"]]
    y = data[classes]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=parameters["test_data_ratio"], random_state=parameters["random_state"]
    )

    # When returning many variables, it is a good practice to give them names:
    return dict(
        train_x=X_train,
        train_y=y_train,
        test_x=X_test,
        test_y=y_test
    )


And the beginning of the `data_science/nodes.py`:

In [ ]:
def train_model(
    train_x: pd.DataFrame, train_y: pd.DataFrame, parameters: dict[str, Any]
) -> np.ndarray:
    num_iter = parameters["num_train_iter"]
    lr = parameters["learning_rate"]
    ...

You also need to modify the both `pipeline.py` files. There should be an
```
"params:model_options"
```
string wherever we need to pass the parameters to a node.

Finally, we used `scikit-learn`, so we need to install it (good practice is to add it right away to `requirements.txt`).
```
pip install scikit-learn
```

Run the `kedro run` command to test if the refactor succeeded.

## Excercise: Taxi Trips

As an excercise, your task is to remake the example project above to the [Taxi Trips dataset](https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew). Let us download and explore this dataset below.

In [1]:
import pandas as pd
import tempfile
import urllib
import os
from pprint import pprint

In [2]:
_data_root = tempfile.mkdtemp(prefix='tfx-data')
DATA_PATH = 'https://raw.githubusercontent.com/tensorflow/tfx/master/tfx/examples/chicago_taxi_pipeline/data/simple/data.csv'
_data_filepath = os.path.join(_data_root, "data.csv")
urllib.request.urlretrieve(DATA_PATH, _data_filepath)

('/tmp/tfx-datalyvlq62g/data.csv', <http.client.HTTPMessage at 0x7943aac12c90>)

In [4]:
print(_data_filepath)
!mv {_data_filepath} data_taxi.csv

/tmp/tfx-datalyvlq62g/data.csv


In [6]:
df = pd.read_csv("data_taxi.csv")

In [7]:
pprint(list(df.columns))
df.head()

['pickup_community_area',
 'fare',
 'trip_start_month',
 'trip_start_hour',
 'trip_start_day',
 'trip_start_timestamp',
 'pickup_latitude',
 'pickup_longitude',
 'dropoff_latitude',
 'dropoff_longitude',
 'trip_miles',
 'pickup_census_tract',
 'dropoff_census_tract',
 'payment_type',
 'company',
 'trip_seconds',
 'dropoff_community_area',
 'tips']


,pickup_community_area,fare,trip_start_month,trip_start_hour,trip_start_day,trip_start_timestamp,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,trip_miles,pickup_census_tract,dropoff_census_tract,payment_type,company,trip_seconds,dropoff_community_area,tips
0,NaN,12.45,5,19,6,1400269500,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Credit Card,Chicago Elite Cab Corp. (Chicago Carriag,0.0,NaN,0.0
1,NaN,0.00,3,19,5,1362683700,NaN,NaN,NaN,NaN,0.0,NaN,NaN,Unknown,Chicago Elite Cab Corp.,300.0,NaN,0.0
2,60.0,27.05,10,2,3,1380593700,41.836150,-87.648788,NaN,NaN,12.6,NaN,NaN,Cash,Taxi Affiliation Services,1380.0,NaN,0.0
3,10.0,5.85,10,1,2,1382319000,41.985015,-87.804532,NaN,NaN,0.0,NaN,NaN,Cash,Taxi Affiliation Services,180.0,NaN,0.0
4,14.0,16.65,5,7,5,1369897200,41.968069,-87.721559,NaN,NaN,0.0,NaN,NaN,Cash,Dispatch Taxi Affiliation,1080.0,NaN,0.0


As you see, the dataset contains a lot of NaNs. In this case we don't want to remove all rows containing a NaN, as it would greatly hinder the size of the dataset, instead we will replace them with 0 or empty string "". Moreover we need to preproces those columns, some of them are bucket features, while other are categorical features, below we already divided them into 4 categories that need different preprocessing. The bucket features should be divided into 10 buckets and 1-hot encoded, while the categorical features whould be 1-hot encoded. The numerical features should be normalized.

The objective is to predict if the tip value is greater than 20% of the fare. Thus, you need also to add such a column to the dataset, remove the tip from the dataset as it would be a proxy column and do a train-test split.

Finally you need to train any model that predicts if the tip is greater than 20% of the fare. You can choose any model that you want (logistic regression is fine).

TIP: It might be easier for you to firstly develop your transformations and model here and then move it to Kedro. For such kind of development Kedro has a jupyter support - you can read about it in the `README.md` file in your project.

In [ ]:
NUMERICAL_FEATURES = [
    'trip_miles',
    'fare',
    'trip_seconds'
    ]

BUCKET_FEATURES = [
    'pickup_latitude',
    'pickup_longitude',
    'dropoff_latitude',
    'dropoff_longitude'
]
FEATURE_BUCKET_COUNT = 10

CATEGORICAL_NUMERICAL_FEATURES = [
    'trip_start_hour',
    'trip_start_day',
    'trip_start_month',
    'pickup_census_tract',
    'dropoff_census_tract',
    'pickup_community_area',
    'dropoff_community_area'
]

CATEGORICAL_STRING_FEATURES = [
    'payment_type',
    'company',
]

# References

- Kedro documentation and tutorals: https://docs.kedro.org/en/0.18.14/index.html
- YouTube playlist with Kedro tutorial: https://www.youtube.com/playlist?list=PL-JJgymPjK5LddZXbIzp9LWurkLGgB-nY